# Checkpoint 1 - Apache Spark
## Tratamento, consulta e qualidade de dados

**Disciplina:** Modern Data Architecture and Engineering

Nome: Filipe Santos de Oliveira  RM: 572828

Nome: Yasmin Yumi Tsunokawa  RM: 569408


---

## Contexto dos dados

Recorte da base publica **Anatel - Utilidade Publica**, com o uso de ferramentas de alertas de desastres disponibilizadas pelas prestadoras de telecomunicacoes as Defesas Civis. Granularidade horaria, atualizacao mensal.

| Campo | Descricao |
|---|---|
| Forma de Envio | SMS - mensagens de texto para celulares (SMP); TVA - mensagens para assinantes de TV por assinatura. |
| Data | Data do envio do alerta. |
| Codigo do Alerta | Codigo do registro do alerta. |
| Mensagem | Mensagem de texto enviada no alerta. |
| Tipo de Alerta | Descricao do tipo de alerta, conforme Cobrade. |
| Municipio | Presente no CSV; sem descricao individualizada no glossario original. |
| Codigo IBGE | Codigo IBGE do municipio de ocorrencia. |
| UF | Sigla da Unidade da Federacao do municipio de ocorrencia. |

Fonte: Agencia Nacional de Telecomunicacoes - Anatel.

## Configuracao do ambiente

In [17]:
!pip install -q pyspark

In [2]:
import os

if not os.path.exists("alertas_1000.csv"):
    from google.colab import files
    uploaded = files.upload()

## Questao 1 - Leitura e inspecao inicial (1,0 ponto)

In [3]:
# Questao 1a
# Cria a SparkSession e carrega o CSV no DataFrame principal 'alertas'.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("AnatelAlertasDefesaCivil").getOrCreate()

alertas = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("encoding", "ISO-8859-1")
    .csv("alertas_1000.csv")
)

In [4]:
# Questao 1b
# Cinco registros e o schema do DataFrame
alertas.show(5, truncate=False)
alertas.printSchema()

+--------------+----------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+-----------------------+-----------+---+----+
|Forma de Envio|Data            |Código do Alerta|Mensagem                                                                                                                                                     |Tipo de Alerta            |Município              |Código IBGE|UF |-   |
+--------------+----------------+----------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+-----------------------+-----------+---+----+
|SMS           |02/06/2026 06:55|18750/2026      |Defesa Civil:(ES) ALERTA de CHUVA, RAIOS, VENTO e GRANIZO PONTUAL na regiao Sul nas proximas 03 horas.Prote

In [5]:
# Questao 1c
# Quantidade total de registros carregados
total_registros = alertas.count()
print(f"Total de registros carregados: {total_registros}")

Total de registros carregados: 1000


**Conclusao (1b/1c):** o arquivo foi lido corretamente, com cabecalho, 9 colunas e caracteres acentuados preservados (ex.: 'Codigo do Alerta', 'Municipio'). O DataFrame `alertas` possui **1000 registros**.

## Questao 2 - Preparacao dos dados (1,5 ponto)

In [6]:
# Questao 2a
# Padroniza os nomes das colunas: minusculas, sem acentos, espacos -> underline
import unicodedata

def padronizar_nome_coluna(nome):
    sem_acento = unicodedata.normalize("NFKD", nome)
    sem_acento = "".join(c for c in sem_acento if not unicodedata.combining(c))
    return sem_acento.strip().lower().replace(" ", "_")

novos_nomes = [padronizar_nome_coluna(c) for c in alertas.columns]
alertas = alertas.toDF(*novos_nomes)
print(alertas.columns)

['forma_de_envio', 'data', 'codigo_do_alerta', 'mensagem', 'tipo_de_alerta', 'municipio', 'codigo_ibge', 'uf', '-']


In [7]:
# Questao 2b
# Verifica se existe coluna sem informacao util (evidencia: valores distintos e contagem de nulos/vazios)
alertas.groupBy("-").count().show()

total = alertas.count()
vazios = alertas.filter((F.col("-").isNull()) | (F.trim(F.col("-")) == "")).count()
print(f"Registros com a coluna '-' vazia/nula: {vazios} de {total}")

+----+-----+
|   -|count|
+----+-----+
|NULL| 1000|
+----+-----+

Registros com a coluna '-' vazia/nula: 1000 de 1000


**Conclusao (2b):** a coluna `-` (ultima coluna do CSV) esta vazia em **todos os 1000 registros** - nao carrega nenhuma informacao util. Ela sera removida do DataFrame.

In [8]:
# Questao 2b (continuacao) - remove a coluna sem informacao util
alertas = alertas.drop("-")
alertas.columns

['forma_de_envio',
 'data',
 'codigo_do_alerta',
 'mensagem',
 'tipo_de_alerta',
 'municipio',
 'codigo_ibge',
 'uf']

In [9]:
# Questao 2c
# Padroniza forma_de_envio e uf (maiusculas + remove espacos excedentes)
# Remove espacos excedentes de municipio e tipo_de_alerta (sem alterar caixa)
def limpar_espacos(coluna):
    return F.trim(F.regexp_replace(F.col(coluna), r"\s+", " "))

alertas = (
    alertas
    .withColumn("forma_de_envio", F.upper(limpar_espacos("forma_de_envio")))
    .withColumn("uf", F.upper(limpar_espacos("uf")))
    .withColumn("municipio", limpar_espacos("municipio"))
    .withColumn("tipo_de_alerta", limpar_espacos("tipo_de_alerta"))
)
alertas.select("forma_de_envio", "uf", "municipio", "tipo_de_alerta").show(5, truncate=False)

+--------------+---+-----------------------+--------------------------+
|forma_de_envio|uf |municipio              |tipo_de_alerta            |
+--------------+---+-----------------------+--------------------------+
|SMS           |ES |Bom Jesus do Norte (ES)|CHUVAS INTENSAS           |
|SMS           |ES |Bom Jesus do Norte (ES)|CHUVAS INTENSAS           |
|SMS           |ES |Bom Jesus do Norte (ES)|DOENÇAS INFECCIOSAS VIRAIS|
|SMS           |ES |Bom Jesus do Norte (ES)|CHUVAS INTENSAS           |
|SMS           |ES |Bom Jesus do Norte (ES)|CHUVAS INTENSAS           |
+--------------+---+-----------------------+--------------------------+
only showing top 5 rows


In [10]:
# Questao 2d
# Converte 'data' (string dd/MM/yyyy HH:mm) para timestamp em uma nova coluna 'data_hora'
alertas = alertas.withColumn(
    "data_hora", F.to_timestamp(F.col("data"), "dd/MM/yyyy HH:mm")
)

nao_convertidos = alertas.filter(F.col("data_hora").isNull()).count()
print(f"Registros que nao puderam ser convertidos para timestamp: {nao_convertidos}")

alertas.select("data", "data_hora").show(5, truncate=False)

Registros que nao puderam ser convertidos para timestamp: 0
+----------------+-------------------+
|data            |data_hora          |
+----------------+-------------------+
|02/06/2026 06:55|2026-06-02 06:55:00|
|06/02/2026 19:11|2026-02-06 19:11:00|
|11/04/2020 17:17|2020-04-11 17:17:00|
|09/06/2025 22:49|2025-06-09 22:49:00|
|23/12/2024 15:01|2024-12-23 15:01:00|
+----------------+-------------------+
only showing top 5 rows


**Conclusao (2d):** **0 registros** falharam na conversao - todos os valores de `data` seguem o padrao `dd/MM/yyyy HH:mm`, entao `data_hora` foi preenchida com sucesso em 100% das linhas.

## Questao 3 - Documentacao e consistencia (1,5 ponto)

In [11]:
# Questao 3a
# Valores distintos de forma_de_envio, para comparar com o dicionario de dados
alertas.groupBy("forma_de_envio").count().orderBy(F.desc("count")).show()

+--------------+-----+
|forma_de_envio|count|
+--------------+-----+
|           SMS|  974|
|           DCA|   20|
|           TVA|    6|
+--------------+-----+



**Conclusao (3a):** o dicionario de dados documenta apenas os valores **SMS** e **TVA** para `forma_de_envio`. No CSV aparece tambem o valor **DCA** (20 registros), que **nao esta documentado** no glossario fornecido. Portanto, nem todos os valores encontrados estao documentados.

In [12]:
# Questao 3b
# Separa codigo_do_alerta (formato numero/ano) em numero_do_alerta e ano_do_codigo (inteiro)
alertas = (
    alertas
    .withColumn("numero_do_alerta", F.split(F.col("codigo_do_alerta"), "/").getItem(0))
    .withColumn("ano_do_codigo", F.split(F.col("codigo_do_alerta"), "/").getItem(1).cast("int"))
)
alertas.select("codigo_do_alerta", "numero_do_alerta", "ano_do_codigo").show(5, truncate=False)

+----------------+----------------+-------------+
|codigo_do_alerta|numero_do_alerta|ano_do_codigo|
+----------------+----------------+-------------+
|18750/2026      |18750           |2026         |
|6609/2026       |6609            |2026         |
|17581/2020      |17581           |2020         |
|11940/2025      |11940           |2025         |
|17865/2024      |17865           |2024         |
+----------------+----------------+-------------+
only showing top 5 rows


In [13]:
# Questao 3c
# Compara ano_do_codigo com o ano de data_hora
alertas = alertas.withColumn("ano_data_hora", F.year(F.col("data_hora")))

divergentes = alertas.filter(F.col("ano_do_codigo") != F.col("ano_data_hora")).count()
print(f"Linhas com divergencia entre ano_do_codigo e o ano de data_hora: {divergentes}")

Linhas com divergencia entre ano_do_codigo e o ano de data_hora: 0


**Conclusao (3c):** foram encontradas **0 divergencias** entre `ano_do_codigo` e o ano de `data_hora` nos 1000 registros. Isso indica que, neste recorte, o codigo do alerta e sempre gerado no mesmo ano em que o alerta e efetivamente enviado - ou seja, os dois campos sao **consistentes** entre si.

## Questao 4 - Consultas com a API de DataFrame (1,5 ponto)

In [14]:
# Questao 4a
# Cinco municipios com maior quantidade de notificacoes, do maior para o menor
(
    alertas.groupBy("municipio")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
    .show(truncate=False)
)

+------------------------+-----+
|municipio               |count|
+------------------------+-----+
|Guaçuí (ES)             |166  |
|Mimoso do Sul (ES)      |160  |
|São José do Calçado (ES)|154  |
|Apiacá (ES)             |152  |
|Bom Jesus do Norte (ES) |150  |
+------------------------+-----+



In [15]:
# Questao 4b
# Quantidade de notificacoes por tipo_de_alerta, do maior para o menor
(
    alertas.groupBy("tipo_de_alerta")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+---------------------------------------------+-----+
|tipo_de_alerta                               |count|
+---------------------------------------------+-----+
|CHUVAS INTENSAS                              |731  |
|DOENÇAS INFECCIOSAS VIRAIS                   |132  |
|DESLIZAMENTOS                                |63   |
|VENDAVAL                                     |30   |
|INUNDAÇÕES                                   |19   |
|GRANIZO                                      |12   |
|ALAGAMENTOS                                  |6    |
|ENXURRADAS                                   |5    |
|TEMPESTADE LOCAL/CONVECTIVA - CHUVAS INTENSAS|1    |
|ONDA DE FRIO - FRIAGEM                       |1    |
+---------------------------------------------+-----+



In [16]:
# Questao 4c
# Codigo_do_alerta, data_hora, tipo_de_alerta e forma_de_envio para um municipio escolhido,
# em ordem cronologica
municipio_escolhido = "Bom Jesus do Norte (ES)"

registros_municipio = (
    alertas.filter(F.col("municipio") == municipio_escolhido)
    .select("codigo_do_alerta", "data_hora", "tipo_de_alerta", "forma_de_envio")
    .orderBy(F.col("data_hora").asc())
)
registros_municipio.show(registros_municipio.count(), truncate=False)

+----------------+-------------------+--------------------------+--------------+
|codigo_do_alerta|data_hora          |tipo_de_alerta            |forma_de_envio|
+----------------+-------------------+--------------------------+--------------+
|1593/2018       |2018-01-13 07:52:00|CHUVAS INTENSAS           |SMS           |
|1669/2018       |2018-01-14 19:25:00|CHUVAS INTENSAS           |SMS           |
|1974/2018       |2018-01-22 09:25:00|CHUVAS INTENSAS           |SMS           |
|2203/2018       |2018-01-29 17:09:00|CHUVAS INTENSAS           |SMS           |
|2242/2018       |2018-02-01 10:36:00|CHUVAS INTENSAS           |SMS           |
|2256/2018       |2018-02-05 11:53:00|CHUVAS INTENSAS           |SMS           |
|2851/2018       |2018-03-08 15:51:00|CHUVAS INTENSAS           |SMS           |
|3096/2018       |2018-03-17 14:17:00|CHUVAS INTENSAS           |SMS           |
|3240/2018       |2018-03-21 15:26:00|CHUVAS INTENSAS           |SMS           |
|3271/2018       |2018-03-22